<a href="https://www.kaggle.com/code/isithadinujaya/deep-audio-zooming-ml-model?scriptVersionId=301477366" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
class SubbandBeamformer(nn.module):
    def __init__(self, num_mics= 8, num_freqs = 257, hidden_size = 128, num_directions = 2):
        #num_freqs = num_fre_bins (n.fft//2 +1)
        #num_directions : D_in, D_out (2_features)

        super().__init__()
        self.num_mics = num_mics
        self.num_freqs
        self.hidden_size = hidden_size

#input vector (LPS, D_IN, D_OUT)

        self.input_size = 3

        #temporal GRU layer

        self.gru1= nn.GRU(input_size = self.input_size, hidden_size = hidden_size, batch_first=True, bidirectional = False)

        #linear layers for masking

        self.mask_out = nn.linear(hidden_size, num_freqs*2)

        #embeddings 
        self.embedding_dim = 6

        self.ln_embed = nn.LayerNorm(self.embedding_dim)
        self.fc_embed = nn.linear(self.embedding_dim, hidden_size) #project too hidden sizes


        self.gru_freq = nn.GRUCell(input_size = hidden_size) #GRUthat runs for each frequency input shape - (batch,time, hidden_size)

        self.fc_filter = nn.linear(hidden_size, num_mics*2) #outputlayer to estimate complex beamforming filters (one per frequency per microphone?)
        # estimate complex enhancement filters W(f)

    def forward(self, lps, D_in, D_out, Y_ref, fov_az_range = None):
        
        batch, freq, time = lps.shape
        device = lps.device

        #GRU expects (batch, seq_len, input_size)
        features = torch.stack([lps, D_in, D_out], dim=1)

        features = features.permute(0,2,3,1)
        #(batch, freq, time, 3)

        features = features.reshape(batch*freq, time, 3)

        #GRU1
        gru_out,_ = self.gru1(features)

        #output at each time step to compute masks per T-F bin

        
        mask_in = self.mask_in(gru_out)


        self.mask_in_fc = nn.linear(hidden_size, 2)

        self.mask_out_fc = nn.linear(hidden_size, 2)

        mask_in_ri  = self.mask_in_fc(gru_out)

        mask_out_ri = self.mask_out_fc(gru_out)

        mask_in = torch.view_as_complex(mask_in_ri.reshape(batch, freq, time,2))

        mask_out = torch.view_as_complex(mask_out_ri.reshape(batch, freq, time,2))

        target_est = mask_in*Y_ref

        interference_est = mask_out*Y_ref
        
        
        
        embed = torch.stack([Y_ref.real, Y_ref.imag, target_est.real, target_est.imag, interference_est.real, interference_est.imag], dim=1)

#layer norm and linear projection
        embed = self.ln_embed(embed)
        embed = self.fc_embed(embed)

        # Per-frequency RNN: we'll treat each frequency independently.
        # We'll loop over time steps using GRU cell.
        # Reshape to (freq, batch, time, hidden_size)
        embed_f = embed.permute(1, 0, 2, 3)  # (freq, batch, time, hidden_size)

        # Initialize hidden state for each freq*batch
        h = torch.zeros(batch * freq, self.hidden_size, device=device)
        outputs = []
        for t in range(time):
            inp_t = embed_f[..., t, :]  # (freq, batch, hidden_size)
            inp_t = inp_t.reshape(-1, self.hidden_size)  # (freq*batch, hidden_size)
            h = self.gru_freq_cell(inp_t, h)
            outputs.append(h.clone())
        # outputs: list of (freq*batch, hidden_size) for each time
        out_f = torch.stack(outputs, dim=1)  # (freq*batch, time, hidden_size)
        out_f = out_f.reshape(freq, batch, time, self.hidden_size).permute(1, 0, 2, 3)  # (batch, freq, time, hidden_size)

        # Estimate filters
        filters_ri = self.fc_filter(out_f)  # (batch, freq, time, num_mics*2)
        filters_ri = filters_ri.reshape(batch, freq, time, self.num_mics, 2)
        filters = torch.view_as_complex(filters_ri)  # (batch, freq, time, num_mics)

        # Apply beamforming
        Y_enhanced = torch.einsum('bftm,bmft->bft', filters.conj(), Y_mix)

        return Y_enhanced, mask_in, mask_out, filters
        


    
        
        
        